# Evaluation: the trained adapters on Colab or Kaggle

Runs `lcs evaluate --skip-paid` with `configs/eval.yaml`. That covers copy, truncate, the base model zero-shot and few-shot, SFT, SFT+DPO and SFT+GRPO, scored on the golden set and dev. It can also run the open-weight judge (Granite 8B) as a cross-check. You get `results/<run>/` back as a zip.

- **GPU:** one 24 GB GPU (L4 / A10G) for generation and the Granite judge. A 16 GB T4 handles generation in fp16; the judge may not fit.
- **Inputs:** `data/processed/`, `data/golden/` and `checkpoints/{sft,dpo,grpo}/adapter/` (RUNBOOK §1–3).
- **Not here:** the Claude system and the Claude judge are paid and need no GPU. Run them from your own machine with `--yes` (RUNBOOK §4). Nothing is pushed to the Hub.

In [ ]:
!nvidia-smi

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/erfunm/Legal-Clause-Simplifier-using-RLHF.git"
BRANCH = "main"
CONFIG = "configs/eval.yaml"

# Inputs: one zip made locally with
#   zip -r eval_inputs.zip data/processed data/golden checkpoints/sft/adapter checkpoints/dpo/adapter checkpoints/grpo/adapter
INPUT_SOURCE = "upload"  # "upload" (Colab) or "kaggle" (a Kaggle dataset holding the unzipped folders)
KAGGLE_INPUT_DIR = "/kaggle/input/legal-clause-eval-inputs"

RUN_HF_JUDGE = False  # the open-weight cross-check judge (about 16 GB in bf16)

## 2. Install

In [ ]:
!git clone --depth 1 --branch {BRANCH} {REPO_URL} repo
%cd repo
!pip install -q -e ".[train,eval]"

## 3. Inputs

In [ ]:
import shutil
import zipfile
from pathlib import Path

if INPUT_SOURCE == "upload":
    from google.colab import files

    uploaded = files.upload()  # choose eval_inputs.zip
    with zipfile.ZipFile(next(iter(uploaded))) as archive:
        archive.extractall(".")
elif INPUT_SOURCE == "kaggle":
    for sub in ("data/processed", "data/golden", "checkpoints"):
        shutil.copytree(Path(KAGGLE_INPUT_DIR) / sub, sub, dirs_exist_ok=True)
else:
    raise ValueError(f"unknown INPUT_SOURCE {INPUT_SOURCE!r}")

from legal_simplifier.config import EvalConfigFile, load_config

cfg = load_config(CONFIG, EvalConfigFile)
assert (cfg.data.golden_dir / "drafts.jsonl").exists(), "missing data/golden"
for system in cfg.systems:
    if system.adapter is not None:
        assert (system.adapter / "adapter_config.json").exists(), f"missing {system.adapter}"
RUN_DIR = cfg.output_root / cfg.experiment.name
print("Will write", RUN_DIR)

## 4. Evaluate the free systems

In [ ]:
!lcs evaluate --config {CONFIG} --skip-paid

## 5. Optional: the open-weight judge, then the error analysis

In [ ]:
if RUN_HF_JUDGE:
    !lcs judge --config {CONFIG} --run {RUN_DIR} --backend hf
!lcs error-analysis --config {CONFIG} --run {RUN_DIR}

In [ ]:
import json

metrics = json.loads((RUN_DIR / "metrics.json").read_text())
for system, per in metrics["systems"].items():
    golden = per.get("golden", {})
    print(system, {k: round(golden[k]["mean"], 3) for k in ("sari", "entity_recall", "nli_faithfulness", "detector/condition_drop") if golden.get(k)})

## 6. Download

Unzip into the repo on your machine (`unzip eval_run.zip`). Then run the Claude system and judge there with `--yes`, followed by `lcs report`. The generation cache comes with it, so nothing is regenerated.

In [ ]:
!zip -qr eval_run.zip {RUN_DIR} results/cache
try:
    from google.colab import files

    files.download("eval_run.zip")
except ImportError:
    print(f"Saved {Path('eval_run.zip').resolve()}")